# 05 · LightGBM global model

Drives `m5.models.lgbm.build_lgbm_forecaster` and `m5.cv.lgbm_cv` — the 
same path `make cv-lgbm` and `make forecast-lgbm` use. Notebook outputs 
match CLI outputs byte-for-byte.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd

from m5.config import SETTINGS, set_global_seed
from m5.cv import lgbm_cv
from m5.evaluation import compute_components, make_submission, wrmsse_for_models
from m5.models.lgbm import (
    DEFAULT_LAGS,
    DEFAULT_ROLLS,
    build_lgbm_forecaster,
    fit_predict_lgbm,
    lgbm_params,
)
from m5.plots import configure_style

configure_style()
set_global_seed()

## Inspect the canonical hyperparameters

Tweedie objective, `deterministic=True`, `force_row_wise=True`, 
`seed=SETTINGS.seed`, lags=(7, 14, 28), rolling means over (7, 28) lagged by 1, 
`Differences([1])`, date_features=`[dayofweek, day, week, month, year]`. 
Two notebook runs produce identical numbers and match the CLI.

In [ ]:
fcst = build_lgbm_forecaster()
params = fcst.models['LGBM'].get_params()
{k: params[k] for k in ('objective', 'tweedie_variance_power', 'learning_rate',
                         'num_leaves', 'n_estimators', 'seed', 'deterministic')}

### To experiment with overrides

Take the canonical defaults and override only the knob you want to vary; 
everything else stays pinned, so the comparison is clean.

In [ ]:
# import lightgbm as lgb
# from mlforecast import MLForecast
# from mlforecast.lag_transforms import RollingMean
#
# custom = {**lgbm_params(), 'num_leaves': 256, 'n_estimators': 2000}
# experimental = MLForecast(
#     models={'LGBM': lgb.LGBMRegressor(**custom)}, freq='D',
#     lags=list(DEFAULT_LAGS),
#     lag_transforms={1: [RollingMean(window_size=w) for w in DEFAULT_ROLLS]},
#     date_features=['dayofweek', 'day', 'week', 'month', 'year'],
# )

## Cross-validation

In [ ]:
df = pd.read_parquet(SETTINGS.processed_dir / 'long.parquet')

cv = lgbm_cv(df, h=SETTINGS.horizon, n_windows=SETTINGS.n_windows)
cv.to_parquet(SETTINGS.artifacts_dir / 'cv_lgbm.parquet', index=False)
cv.tail()

In [ ]:
components = compute_components(df[df['ds'] < cv['ds'].min()])
wrmsse_for_models(cv[['unique_id', 'ds', 'y']], cv, components)

## Train on full data + emit forecast

In [ ]:
forecast = fit_predict_lgbm(df, horizon=SETTINGS.horizon)
forecast.to_parquet(SETTINGS.forecasts_dir / 'forecast_lgbm.parquet', index=False)
forecast.head()

## Optional — Kaggle-format submission

In [ ]:
submission = make_submission(forecast, h=SETTINGS.horizon, value_col='LGBM', layout='kaggle')
submission.head()